### Population England and Spain per year

In [73]:
import pandas as pd

df_en = pd.read_csv('../Extacted data/Population/population_england2020-2024.csv')
xls = pd.read_excel('../Extacted data/Population/population_spain2020-2026.xlsx', sheet_name=None)
print(xls.keys())

dict_keys(['Spain Population'])


In [74]:
df_sp = xls['Spain Population']
df_sp

,Year,Population,Yearly % Change,Yearly Change,Migrants (net),Median Age,Fertility Rate,Density (P/km²),Urban Pop %,Urban Population,World Share,World Population,Global Rank
0,2026,47850793,-0.0008,-39165,85305,46.3,1.24,96,0.796,38082079,0.0058,8300678395,34
1,2025,47889958,-0.0004,-20568,96630,45.9,1.23,96,0.793,37995950,0.0058,8231613070,32
2,2024,47910526,0.0000,-1053,111674,45.4,1.22,96,0.791,37911965,0.0059,8161972572,32
3,2023,47911579,0.0017,83197,119099,44.9,1.21,96,0.790,37829585,0.0059,8091734930,32
4,2022,47828382,0.0016,74447,299779,44.4,1.21,96,0.789,37744345,0.0060,8021407192,31
5,2020,47679489,0.0042,199161,236854,43.5,1.19,96,0.787,37543537,0.0060,7887001292,30


In [75]:
df_en = df_en.drop(columns=["Yearly Change","Net Migration","Median Age","Fertility Rate"])
df_sp = df_sp.drop(columns=['Yearly % Change','Yearly Change','Migrants (net)','Median Age','Fertility Rate','Density (P/km²)','Urban Pop %','Urban Population','World Share','World Population','Global Rank'])

In [76]:
# add new row
df_en.loc[len(df_en)] = [2025, 59050000]
df_en = df_en.drop(0)
df_en

,Year,Population
1,2021,56489800
2,2022,57106398
3,2023,57690300
4,2024,58397300
5,2025,59050000


In [77]:
df_sp.loc[len(df_sp)] = [2021, 47753936]
df_sp = df_sp.drop(0)
df_sp = df_sp.drop(5)
df_sp

,Year,Population
1,2025,47889958
2,2024,47910526
3,2023,47911579
4,2022,47828382
6,2021,47753936


In [78]:
# df_sp = df_sp.drop(0)
# df_en = df_en.drop(0)

In [79]:
df_en['country']= 'England'
df_sp['country']= 'Spain'

In [80]:
df_sp = df_sp.sort_values("Year", ascending=True)
df_sp = df_sp.reset_index(drop=True)
df_en = df_en.reset_index(drop=True)

In [81]:
df_2 = pd.concat([df_en, df_sp])

In [82]:
df_2.sort_values("Year", ascending=True)
df_2.reset_index()
df_2

,Year,Population,country
0,2021,56489800,England
1,2022,57106398,England
2,2023,57690300,England
3,2024,58397300,England
4,2025,59050000,England
0,2021,47753936,Spain
1,2022,47828382,Spain
2,2023,47911579,Spain
3,2024,47910526,Spain
4,2025,47889958,Spain


In [60]:
df_2.to_csv("Population_21_25_all.csv", index=False)

In [ ]:
# import numpy as np

# # Create monthly dates for each country
# def expand_to_monthly(df_pop):
#     monthly_records = []
    
#     for country in df_pop['country'].unique():
#         df_country = df_pop[df_pop['country'] == country].sort_values('Year')
        
#         for i in range(len(df_country) - 1):
#             year_start = df_country.iloc[i]['Year']
#             year_end = df_country.iloc[i + 1]['Year']
#             pop_start = df_country.iloc[i]['Population']
#             pop_end = df_country.iloc[i + 1]['Population']
            
#             # 12 months between years
#             for month in range(1, 13):
#                 fraction = (month - 1) / 12
#                 population = pop_start + (pop_end - pop_start) * fraction
#                 monthly_records.append({
#                     'date': f'{year_start}-{month:02d}',
#                     'population': round(population),
#                     'country': country
#                 })
        
#         # Add last year December
#         last = df_country.iloc[-1]
#         monthly_records.append({
#             'date': f"{last['Year']}-12",
#             'population': round(last['Population']),
#             'country': last['country']
#         })
    
#     return pd.DataFrame(monthly_records)

# df_monthly_pop = expand_to_monthly(df_2)
# df_monthly_pop

,date,population,country
0,2021-01,56489800,England
1,2021-02,56541183,England
2,2021-03,56592566,England
3,2021-04,56643950,England
4,2021-05,56695333,England
...,...,...,...
93,2024-09,47896814,Spain
94,2024-10,47895100,Spain
95,2024-11,47893386,Spain
96,2024-12,47891672,Spain


In [89]:
import numpy as np

def expand_to_monthly(df_pop):
    monthly_records = []
    
    for country in df_pop['country'].unique():
        df_country = df_pop[df_pop['country'] == country].sort_values('Year')
        
        # Calculate average annual growth rate
        populations = df_country['Population'].values
        annual_growth_rates = [(populations[i+1] - populations[i]) / populations[i] 
                               for i in range(len(populations)-1)]
        avg_growth_rate = np.mean(annual_growth_rates)
        
        for i in range(len(df_country) - 1):
            year_start = df_country.iloc[i]['Year']
            pop_start = df_country.iloc[i]['Population']
            pop_end = df_country.iloc[i + 1]['Population']
            
            for month in range(1, 13):
                fraction = (month - 1) / 12
                population = pop_start + (pop_end - pop_start) * fraction
                monthly_records.append({
                    'date': f'{year_start}-{month:02d}',
                    'population': round(population),
                    'country': country
                })
        
        # Generate all 12 months for last year (2025) using growth rate
        last = df_country.iloc[-1]
        last_pop = last['Population']
        last_year = last['Year']
        projected_next_pop = last_pop * (1 + avg_growth_rate)
        
        for month in range(1, 13):
            fraction = (month - 1) / 12
            population = last_pop + (projected_next_pop - last_pop) * fraction
            monthly_records.append({
                'date': f'{last_year}-{month:02d}',
                'population': round(population),
                'country': country
            })
    
    return pd.DataFrame(monthly_records)

df_monthly_pop = expand_to_monthly(df_2)
df_monthly_pop.tail(12)

,date,population,country
108,2025-01,47889958,Spain
109,2025-02,47892799,Spain
110,2025-03,47895639,Spain
111,2025-04,47898480,Spain
112,2025-05,47901321,Spain
113,2025-06,47904161,Spain
114,2025-07,47907002,Spain
115,2025-08,47909842,Spain
116,2025-09,47912683,Spain
117,2025-10,47915524,Spain


In [88]:
df_monthly_pop.to_csv("Population_monthly_21_25_all.csv", index=False)

### Population England per city per year

In [64]:
xls_c = pd.read_excel('../Extacted data/Population/england_city_populations_long_table.xlsx', sheet_name=None)
print(xls_c.keys())

dict_keys(['Long_Table'])


In [65]:
df_c = xls_c['Long_Table']
df_c

,City,Year,Population
0,London,2021,8799800
1,London,2022,8870000
2,London,2023,8945310
3,London,2024,9020000
4,London,2025,9095000
...,...,...,...
80,Kendal,2021,28940
81,Kendal,2022,29000
82,Kendal,2023,29100
83,Kendal,2024,29200


In [87]:
def expand_to_monthly_city(df_pop):
    monthly_records = []
    
    for city in df_pop['City'].unique():
        df_city = df_pop[df_pop['City'] == city].sort_values('Year')
        
        for i in range(len(df_city) - 1):
            year_start = df_city.iloc[i]['Year']
            pop_start = df_city.iloc[i]['Population']
            pop_end = df_city.iloc[i + 1]['Population']
            
            for month in range(1, 13):
                fraction = (month - 1) / 12
                population = pop_start + (pop_end - pop_start) * fraction
                monthly_records.append({
                    'date': f'{year_start}-{month:02d}',
                    'city': city,
                    'population': round(population)
                })
        
        # Add last year December
        last = df_city.iloc[-1]
        monthly_records.append({
            'date': f"{last['Year']}-12",
            'city': last['City'],
            'population': round(last['Population'])
        })
    
    return pd.DataFrame(monthly_records)

df_monthly_city = expand_to_monthly_city(df_c)
df_monthly_city.tail(15)


,date,city,population
818,2023-11,Kendal,29183
819,2023-12,Kendal,29192
820,2024-01,Kendal,29200
821,2024-02,Kendal,29212
822,2024-03,Kendal,29225
823,2024-04,Kendal,29238
824,2024-05,Kendal,29250
825,2024-06,Kendal,29262
826,2024-07,Kendal,29275
827,2024-08,Kendal,29288


In [67]:
df_monthly_city.to_csv("England_city_population_monthly_21_25.csv", index=False)